In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Loading data
train=pd.read_csv("train.csv")
test=pd.read_csv("test.csv")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (77299, 11)
Test shape: (41778, 10)


In [3]:
# Feature Engineering
def engineering_features(df):
  df=df.copy()
  df['hour']=df['timestamp'].str.split(":").str[0].astype(int)
  df['minute']=df['timestamp'].str.split(":").str[1].astype(int)
  df['time_of_day']=df['hour'] * 60 + df['minute']

  # Cyclical encoding of hour (captures 23→0 continuity)
  df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
  df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

  # Day cyclical (assuming weekly pattern, day mod 7)
  df['day_sin'] = np.sin(2 * np.pi * (df['day'] % 7) / 7)
  df['day_cos'] = np.cos(2 * np.pi * (df['day'] % 7) / 7)
  df['geo_prefix'] = df['geohash'].str[:4]
  df['geohash_len'] = df['geohash'].str.len()

  # Temperature: fill missing with median
  df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())

  # RoadType: filling missing with 'Unknown'
  df['RoadType'] = df['RoadType'].fillna('Unknown')

  road_map = {
    'Unknown':0,
    'Residential':1,
    'Street':2,
    'Highway':3
  }
  df['RoadType_enc'] = df['RoadType'].map(road_map).fillna(0).astype(int)

  df['LargeVehicles_enc'] = (df['LargeVehicles'] == 'Allowed').astype(int)
  df['Landmarks_enc'] = (df['Landmarks'] == 'Yes').astype(int)

  # Weather: fill missing + encode
  df['Weather'] = df['Weather'].fillna('Unknown')
  weather_map = {
      'Unknown': 0,
      'Sunny': 1,
      'Cloudy': 2,
      'Rainy': 3,
      'Foggy': 4,
      'Snowy': 5
  }
  df['Weather_enc'] = df['Weather'].map(weather_map).fillna(0).astype(int)

  # Rush hour flag
  df['is_rush_hour'] = ((df['hour'].between(7, 9)) | (df['hour'].between(17, 19))).astype(int)

  # Night flag
  df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
  return df

In [4]:
train = engineering_features(train)
test = engineering_features(test)

In [5]:
geo_mean = train.groupby('geohash')['demand'].mean().rename('geo_demand_mean')
geo_std  = train.groupby('geohash')['demand'].std().rename('geo_demand_std')
train = train.join(geo_mean, on='geohash')
train = train.join(geo_std,  on='geohash')
test  = test.join(geo_mean,  on='geohash')
test  = test.join(geo_std,   on='geohash')
train['geo_demand_mean'] = train['geo_demand_mean'].fillna(train['demand'].mean())
train['geo_demand_std']  = train['geo_demand_std'].fillna(train['demand'].std())
test['geo_demand_mean']  = test['geo_demand_mean'].fillna(train['demand'].mean())
test['geo_demand_std']   = test['geo_demand_std'].fillna(train['demand'].std())
prefix_mean = train.groupby('geo_prefix')['demand'].mean().rename('prefix_demand_mean')
train = train.join(prefix_mean, on='geo_prefix')
test  = test.join(prefix_mean,  on='geo_prefix')
train['prefix_demand_mean'] = train['prefix_demand_mean'].fillna(train['demand'].mean())
test['prefix_demand_mean']  = test['prefix_demand_mean'].fillna(train['demand'].mean())

print("Features engineered.")

Features engineered.


In [6]:
FEATURES = [
    'day', 'hour', 'minute', 'time_of_day',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'NumberofLanes', 'Temperature',
    'RoadType_enc', 'LargeVehicles_enc', 'Landmarks_enc', 'Weather_enc',
    'is_rush_hour', 'is_night',
    'geo_demand_mean', 'geo_demand_std', 'prefix_demand_mean',
    'geohash_len'
]

X = train[FEATURES].values
y = train['demand'].values
X_test = test[FEATURES].values
print(f"\nFeature matrix: {X.shape}, Target: {y.shape}")


Feature matrix: (77299, 20), Target: (77299,)


In [7]:
models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=7,subsample=0.8, colsample_bytree=0.8,random_state=42, n_jobs=-1, verbosity=0),
    "LightGBM": lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, num_leaves=63,subsample=0.8, colsample_bytree=0.8,random_state=42, n_jobs=-1, verbose=-1),
}

In [8]:
results = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    mean_r2 = cv_scores.mean()
    std_r2  = cv_scores.std()
    results[name] = {'mean_r2': mean_r2, 'std_r2': std_r2, 'scores': cv_scores}
    print(f"  {name:<25}  R²={mean_r2:.5f} ± {std_r2:.5f}")

best_name = max(results, key=lambda k: results[k]['mean_r2'])
print(f"\n✅ Best model: {best_name}  (R²={results[best_name]['mean_r2']:.5f})")

  Ridge Regression           R²=0.77877 ± 0.09541
  Random Forest              R²=0.87316 ± 0.07403
  XGBoost                    R²=0.86967 ± 0.08261
  LightGBM                   R²=0.87328 ± 0.08636

✅ Best model: LightGBM  (R²=0.87328)


In [9]:
best_model = models[best_name]
best_model.fit(X, y)
preds = best_model.predict(X_test)
preds = np.clip(preds, 0, 1)

In [ ]:
submission = pd.DataFrame({'Index': test['Index'], 'demand': preds})
submission.to_csv('Submission.csv', index=False)
print(f"Submission saved. Shape: {submission.shape}")
print(submission.head())

Submission saved. Shape: (41778, 2)
   Index    demand
0      0  0.053790
1      1  0.026255
2      2  0.000000
3      3  0.028746
4      4  0.051807


In [ ]:
import json
summary = {name: {'mean_r2': round(v['mean_r2'], 6), 'std_r2': round(v['std_r2'], 6),'scores': [round(s, 6) for s in v['scores'].tolist()]}for name, v in results.items()}
with open('results.json', 'w') as f:
    json.dump({'summary': summary, 'best': best_name, 'features': FEATURES}, f)

print("\nDone!")


Done!
